In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
#    for filename in filenames:
#        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# [Cell 2] Setup and config
from pathlib import Path
import gc
import json
import math
import random
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import optuna
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# -----------------------
# Experiment configuration
# -----------------------
SEED = 42
BACKBONES = ["densenet121", "resnet50", "efficientnet_b0"]   # shorten this list to save GPU budget
IMAGE_SIZE = 224
NUM_WORKERS = 2
USE_IMAGENET_WEIGHTS = True
EVAL_BATCH_SIZE = 64        # eval has no gradients, so a bigger batch is faster and does not change results
MIN_POS = 10                # min positives AND negatives needed to report a label's AUROC

LABELS = ["Effusion", "Cardiomegaly", "Pneumothorax", "Atelectasis",
          "Consolidation", "Mass", "Nodule"]

# Tuning split: 1/TUNE_SPLITS of the official train_val patients -> tuning (~11%)
TUNE_SPLITS = 9

# Stage 1: light-sample search
SEARCH_TRAIN_SIZE = 10_000
SEARCH_VAL_SIZE = 4_000
N_TRIALS = 8
TUNING_EPOCHS = 4

# Stage 2: final retrain of the winning config (sample sizes are capped at what the split holds)
FINAL_TRAIN_SIZE = 40_000
FINAL_VAL_SIZE = 8_000
FINAL_EPOCHS = 5

STUDY_TAG = "v1"            # change to start a fresh Optuna study

# Output of dsai-nih-full-preprocessing, attached as an input to this notebook
PREPROCESSED_ROOT = Path("/kaggle/input/notebooks/t1f3001843/dsai-nih-preprocessing")
OUTPUT_DIR = Path("/kaggle/working/nih_multi_backbone_tuning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()

Device: cuda
GPU: Tesla T4


In [3]:
# [Cell 3] Load manifest and resolve shard paths
manifest_path = PREPROCESSED_ROOT / "outputs/processed/nih_full_manifest.csv"
if not manifest_path.exists():
    candidates = list(PREPROCESSED_ROOT.rglob("nih_full_manifest.csv"))
    if not candidates:
        raise FileNotFoundError(f"Could not find nih_full_manifest.csv under {PREPROCESSED_ROOT}")
    manifest_path = candidates[0]

print(f"Manifest: {manifest_path}")
manifest = pd.read_csv(manifest_path)

required_columns = {"official_split", "patient_id", "shard_index", "shard_path", *LABELS}
missing_columns = required_columns - set(manifest.columns)
if missing_columns:
    raise ValueError(f"Manifest is missing columns: {sorted(missing_columns)}")

# One-time index of .npy shard files, keyed by (export_name, split, filename)
shard_files_by_key = {}
for path in PREPROCESSED_ROOT.rglob("*.npy"):
    shard_files_by_key[path.parts[-3:]] = path


def resolve_shard_path(value):
    candidate = Path(str(value))
    if candidate.is_file():
        return str(candidate)
    match = shard_files_by_key.get(candidate.parts[-3:])
    return str(match) if match is not None else None


manifest["shard_path"] = manifest["shard_path"].map(resolve_shard_path)
missing_shards = manifest["shard_path"].isna().sum()
if missing_shards:
    raise FileNotFoundError(f"Could not resolve {missing_shards:,} manifest rows to .npy shards.")

manifest["shard_index"] = pd.to_numeric(manifest["shard_index"], errors="coerce")
if manifest["shard_index"].isna().any():
    raise ValueError("Some manifest rows have a missing/non-numeric shard_index.")
manifest["shard_index"] = manifest["shard_index"].astype(int)
manifest["official_split"] = manifest["official_split"].astype(str).str.lower()
manifest["patient_id"] = manifest["patient_id"].astype(str)

for c in LABELS:
    if not manifest[c].isin([0, 1]).all():
        raise ValueError(f"Label column {c} has values other than 0/1.")

train_all = manifest[manifest["official_split"].eq("train_val")].copy()
test_all = manifest[manifest["official_split"].eq("test")].copy()
if train_all.empty or test_all.empty:
    raise ValueError(f"Expected non-empty train_val and test splits; got {len(train_all)} and {len(test_all)}. "
                     f"Splits present: {manifest['official_split'].unique().tolist()}")

print(manifest["official_split"].value_counts())
print(f"train_val: {len(train_all):,} images / {train_all['patient_id'].nunique():,} patients")
print(f"test     : {len(test_all):,} images / {test_all['patient_id'].nunique():,} patients")

Manifest: /kaggle/input/notebooks/t1f3001843/dsai-nih-preprocessing/outputs/processed/nih_full_manifest.csv
official_split
train_val    25045
test         11778
Name: count, dtype: int64
train_val: 25,045 images / 9,127 patients
test     : 11,778 images / 1,169 patients


In [4]:
# [Cell 4] Patient-level splits
RARE_FIRST = train_all[LABELS].sum().sort_values().index.tolist()   # rarest label first
print("Rarity order (rarest first):", RARE_FIRST)


def strata(df):
    """One stratum per row: the rarest positive label it has, or a final 'none' stratum."""
    pos = np.column_stack([df[c].eq(1).to_numpy() for c in RARE_FIRST])
    return np.where(pos.any(axis=1), pos.argmax(axis=1), len(RARE_FIRST))


def patient_holdout(df, n_splits):
    """Hold out ~1/n_splits of PATIENTS, stratified on rare labels. Returns (kept, held_out)."""
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    keep, hold = next(sgkf.split(df, strata(df), groups=df["patient_id"]))
    return df.iloc[keep].reset_index(drop=True), df.iloc[hold].reset_index(drop=True)


train_data, val_tune = patient_holdout(train_all, n_splits=TUNE_SPLITS)
test_data = test_all.reset_index(drop=True)

split_sets = {"train": train_data, "tune": val_tune, "test": test_data}
names = list(split_sets)
for i, a in enumerate(names):
    for b in names[i + 1:]:
        shared = set(split_sets[a]["patient_id"]) & set(split_sets[b]["patient_id"])
        assert not shared, f"Patient overlap between {a} and {b}: {len(shared)} patients"

for name, d in split_sets.items():
    positives = {c: int(d[c].sum()) for c in RARE_FIRST}
    print(f"{name:6s} n={len(d):>8,}  patients={d['patient_id'].nunique():>7,}  positives={positives}")

Rarity order (rarest first): ['Cardiomegaly', 'Pneumothorax', 'Consolidation', 'Mass', 'Nodule', 'Atelectasis', 'Effusion']
train  n=  22,319  patients=  8,117  positives={'Cardiomegaly': 1500, 'Pneumothorax': 2332, 'Consolidation': 2536, 'Mass': 3625, 'Nodule': 4226, 'Atelectasis': 7351, 'Effusion': 7711}
tune   n=   2,726  patients=  1,010  positives={'Cardiomegaly': 207, 'Pneumothorax': 305, 'Consolidation': 316, 'Mass': 409, 'Nodule': 482, 'Atelectasis': 929, 'Effusion': 948}
test   n=  11,778  patients=  1,169  positives={'Cardiomegaly': 1069, 'Pneumothorax': 2665, 'Consolidation': 1815, 'Mass': 1748, 'Nodule': 1623, 'Atelectasis': 3279, 'Effusion': 4658}


In [5]:
# [Cell 5] Label diagnostics
def label_diagnostics(df, name):
    rows = []
    for c in LABELS:
        pos = int(df[c].eq(1).sum())
        rows.append({"label": c, "positive": pos, "negative": len(df) - pos,
                     "positive_rate": f"{pos / len(df):.2%}"})
    n_labels = df[LABELS].sum(axis=1)
    print(f"\n{name} (n={len(df):,}) | mean target labels per image: {n_labels.mean():.2f} | "
          f"images with 2+ labels: {(n_labels >= 2).mean():.1%} | images with none: {(n_labels == 0).mean():.1%}")
    display(pd.DataFrame(rows).set_index("label"))


for name, d in split_sets.items():
    label_diagnostics(d, name)


train (n=22,319) | mean target labels per image: 1.31 | images with 2+ labels: 25.8% | images with none: 0.0%


,positive,negative,positive_rate
label,,,
Effusion,7711,14608,34.55%
Cardiomegaly,1500,20819,6.72%
Pneumothorax,2332,19987,10.45%
Atelectasis,7351,14968,32.94%
Consolidation,2536,19783,11.36%
Mass,3625,18694,16.24%
Nodule,4226,18093,18.93%



tune (n=2,726) | mean target labels per image: 1.32 | images with 2+ labels: 26.0% | images with none: 0.0%


,positive,negative,positive_rate
label,,,
Effusion,948,1778,34.78%
Cardiomegaly,207,2519,7.59%
Pneumothorax,305,2421,11.19%
Atelectasis,929,1797,34.08%
Consolidation,316,2410,11.59%
Mass,409,2317,15.00%
Nodule,482,2244,17.68%



test (n=11,778) | mean target labels per image: 1.43 | images with 2+ labels: 34.0% | images with none: 0.0%


,positive,negative,positive_rate
label,,,
Effusion,4658,7120,39.55%
Cardiomegaly,1069,10709,9.08%
Pneumothorax,2665,9113,22.63%
Atelectasis,3279,8499,27.84%
Consolidation,1815,9963,15.41%
Mass,1748,10030,14.84%
Nodule,1623,10155,13.78%


In [6]:
# [Cell 6] Float targets and light samples for the two stages
for d in split_sets.values():
    for c in LABELS:
        d[c] = d[c].astype(np.float32)


def sample_frame(df, n):
    return df.sample(n=min(n, len(df)), random_state=SEED).reset_index(drop=True)


search_train = sample_frame(train_data, SEARCH_TRAIN_SIZE)
search_val = sample_frame(val_tune, SEARCH_VAL_SIZE)
final_train = sample_frame(train_data, FINAL_TRAIN_SIZE)
final_val = sample_frame(val_tune, FINAL_VAL_SIZE)

for name, d in [("search_train", search_train), ("search_val", search_val),
                ("final_train", final_train), ("final_val", final_val)]:
    pos = {c: int(d[c].sum()) for c in LABELS}
    print(f"{name:13s} n={len(d):>7,}  positives={pos}")

search_train  n= 10,000  positives={'Effusion': 3436, 'Cardiomegaly': 684, 'Pneumothorax': 1058, 'Atelectasis': 3300, 'Consolidation': 1142, 'Mass': 1593, 'Nodule': 1920}
search_val    n=  2,726  positives={'Effusion': 948, 'Cardiomegaly': 207, 'Pneumothorax': 305, 'Atelectasis': 929, 'Consolidation': 316, 'Mass': 409, 'Nodule': 482}
final_train   n= 22,319  positives={'Effusion': 7711, 'Cardiomegaly': 1500, 'Pneumothorax': 2332, 'Atelectasis': 7351, 'Consolidation': 2536, 'Mass': 3625, 'Nodule': 4226}
final_val     n=  2,726  positives={'Effusion': 948, 'Cardiomegaly': 207, 'Pneumothorax': 305, 'Atelectasis': 929, 'Consolidation': 316, 'Mass': 409, 'Nodule': 482}


In [7]:
# [Cell 7] Epoch and pass accounting
def n_steps(n, batch_size):
    return math.ceil(n / batch_size)


stages = [
    ("Search (per trial)", TUNING_EPOCHS, len(search_train), len(search_val)),
    ("Final retrain", FINAL_EPOCHS, len(final_train), len(final_val)),
]

rows = []
for stage, epochs, n_train, n_val in stages:
    for epoch in range(1, epochs + 1):
        rows.append({
            "stage": stage, "epoch": epoch,
            "train_images": n_train,
            "steps_bs16": n_steps(n_train, 16), "steps_bs32": n_steps(n_train, 32),
            "val_images": n_val, "val_batches": n_steps(n_val, EVAL_BATCH_SIZE),
            "cum_train_images": n_train * epoch, "cum_val_images": n_val * epoch,
        })
display(pd.DataFrame(rows))

search_train_passes = N_TRIALS * TUNING_EPOCHS * len(search_train)
search_val_passes = N_TRIALS * TUNING_EPOCHS * len(search_val)
final_train_passes = FINAL_EPOCHS * len(final_train)
final_val_passes = FINAL_EPOCHS * len(final_val)
test_passes = len(test_data)
n_back = len(BACKBONES)

print(f"\nWorst case per backbone ({N_TRIALS} trials x {TUNING_EPOCHS} epochs search, then {FINAL_EPOCHS}-epoch final):")
print(f"  search : train passes {search_train_passes:>10,} | validation passes {search_val_passes:>9,}")
print(f"  final  : train passes {final_train_passes:>10,} | validation passes {final_val_passes:>9,}")
print(f"  test   : one-off passes {test_passes:,}")
print(f"  total  : train passes {search_train_passes + final_train_passes:>10,} | "
      f"eval passes {search_val_passes + final_val_passes + test_passes:>9,}")
print(f"\nAll {n_back} backbone(s): train passes {n_back * (search_train_passes + final_train_passes):,} | "
      f"eval passes {n_back * (search_val_passes + final_val_passes + test_passes):,}")

,stage,epoch,train_images,steps_bs16,steps_bs32,val_images,val_batches,cum_train_images,cum_val_images
0,Search (per trial),1,10000,625,313,2726,43,10000,2726
1,Search (per trial),2,10000,625,313,2726,43,20000,5452
2,Search (per trial),3,10000,625,313,2726,43,30000,8178
3,Search (per trial),4,10000,625,313,2726,43,40000,10904
4,Final retrain,1,22319,1395,698,2726,43,22319,2726
5,Final retrain,2,22319,1395,698,2726,43,44638,5452
6,Final retrain,3,22319,1395,698,2726,43,66957,8178
7,Final retrain,4,22319,1395,698,2726,43,89276,10904
8,Final retrain,5,22319,1395,698,2726,43,111595,13630



Worst case per backbone (8 trials x 4 epochs search, then 5-epoch final):
  search : train passes    320,000 | validation passes    87,232
  final  : train passes    111,595 | validation passes    13,630
  test   : one-off passes 11,778
  total  : train passes    431,595 | eval passes   112,640

All 3 backbone(s): train passes 1,294,785 | eval passes 337,920


In [8]:
# [Cell 8] Shard-backed dataset
class NIHShardDataset(Dataset):
    MEAN, STD = 0.449, 0.226   # channel-averaged ImageNet statistics

    def __init__(self, frame, training=False):
        self.frame = frame.reset_index(drop=True)
        self.paths = self.frame["shard_path"].astype(str).to_numpy()
        self.positions = self.frame["shard_index"].astype(int).to_numpy()
        self.targets = self.frame[LABELS].to_numpy(dtype=np.float32)
        self.shard_cache = {}   # one cache per DataLoader worker
        self.augmentation = transforms.Compose([
            transforms.RandomAffine(degrees=5, translate=(0.02, 0.02)),
            transforms.ColorJitter(contrast=0.05),
        ]) if training else None   # no flips: left/right matters for findings such as cardiomegaly

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        path, pos = self.paths[index], int(self.positions[index])
        if path not in self.shard_cache:
            self.shard_cache[path] = np.load(path, mmap_mode="r", allow_pickle=False)
        shard = self.shard_cache[path]
        if not 0 <= pos < len(shard):
            raise IndexError(f"shard_index {pos} is outside shard of length {len(shard)}: {path}")

        image = np.asarray(shard[pos])
        if image.ndim != 2:
            raise ValueError(f"Expected grayscale (H, W) image; got {image.shape}")
        image = image.astype(np.float32)
        if shard.dtype == np.uint8:        # decide by dtype, not by this image's max
            image /= 255.0

        image = torch.from_numpy(image.copy()).unsqueeze(0)
        if self.augmentation is not None:
            image = self.augmentation(image)
        image = (image - self.MEAN) / self.STD

        return image, torch.from_numpy(self.targets[index])


def make_loader(frame, batch_size, training):
    return DataLoader(
        NIHShardDataset(frame, training=training),
        batch_size=batch_size,
        shuffle=training,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS > 0),
    )


# Sanity check on one sample
ex_image, ex_targets = NIHShardDataset(search_train)[0]
print("Image:", tuple(ex_image.shape), ex_image.dtype)
print("Normalised pixel range:", round(float(ex_image.min()), 3), "to", round(float(ex_image.max()), 3))
print("Targets:", dict(zip(LABELS, ex_targets.tolist())))
if tuple(ex_image.shape[-2:]) != (IMAGE_SIZE, IMAGE_SIZE):
    print(f"WARNING: shard images are {tuple(ex_image.shape[-2:])}, not {IMAGE_SIZE}x{IMAGE_SIZE}. "
          "Models will still run, but update IMAGE_SIZE or the preprocessing if this is unintended.")

Image: (1, 224, 224) torch.float32
Normalised pixel range: -1.987 to 2.438
Targets: {'Effusion': 0.0, 'Cardiomegaly': 1.0, 'Pneumothorax': 0.0, 'Atelectasis': 0.0, 'Consolidation': 0.0, 'Mass': 0.0, 'Nodule': 0.0}


In [9]:
# [Cell 9] Inspect ten training samples
inspect_ds = NIHShardDataset(search_train)
for i in range(10):
    image, targets = inspect_ds[i]
    found = [l for l, t in zip(LABELS, targets.tolist()) if t == 1] or ["(none of the 7)"]
    print(f"Image {i}: {', '.join(found)}")

Image 0: Cardiomegaly
Image 1: Nodule
Image 2: Effusion, Atelectasis
Image 3: Mass
Image 4: Effusion
Image 5: Nodule
Image 6: Consolidation
Image 7: Atelectasis
Image 8: Effusion, Nodule
Image 9: Atelectasis, Mass


In [10]:
# [Cell 10] Models, BCE loss, AUROC and bootstrap helpers
def adapt_first_conv_to_grayscale(old_conv):
    """Preserve pretrained RGB filters by averaging their input channels."""
    new_conv = nn.Conv2d(
        1, old_conv.out_channels,
        kernel_size=old_conv.kernel_size, stride=old_conv.stride,
        padding=old_conv.padding, dilation=old_conv.dilation,
        groups=old_conv.groups, bias=old_conv.bias is not None,
    )
    with torch.no_grad():
        new_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
        if old_conv.bias is not None:
            new_conv.bias.copy_(old_conv.bias)
    return new_conv


def build_model(backbone, dropout):
    n_out = len(LABELS)
    if backbone == "densenet121":
        weights = models.DenseNet121_Weights.IMAGENET1K_V1 if USE_IMAGENET_WEIGHTS else None
        model = models.densenet121(weights=weights)
        model.features.conv0 = adapt_first_conv_to_grayscale(model.features.conv0)
        model.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.classifier.in_features, n_out))
    elif backbone == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V1 if USE_IMAGENET_WEIGHTS else None
        model = models.resnet50(weights=weights)
        model.conv1 = adapt_first_conv_to_grayscale(model.conv1)
        model.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.fc.in_features, n_out))
    elif backbone == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if USE_IMAGENET_WEIGHTS else None
        model = models.efficientnet_b0(weights=weights)
        model.features[0][0] = adapt_first_conv_to_grayscale(model.features[0][0])
        model.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.classifier[-1].in_features, n_out))
    else:
        raise ValueError(f"Unknown backbone: {backbone}")
    return model


def bce_loss(logits, targets):
    return nn.functional.binary_cross_entropy_with_logits(logits, targets)


def per_label_auroc(probs, targets):
    """AUROC per label (array, NaN where a label has < MIN_POS positives or negatives)."""
    out = np.full(len(LABELS), np.nan)
    for j in range(len(LABELS)):
        y = targets[:, j]
        if (y == 1).sum() < MIN_POS or (y == 0).sum() < MIN_POS:
            continue
        out[j] = roc_auc_score(y, probs[:, j])
    return out


def calculate_auroc(logits, targets):
    probs = torch.sigmoid(logits).numpy()
    aucs = per_label_auroc(probs, targets.numpy())
    scores = {label: float(a) for label, a in zip(LABELS, aucs)}
    finite = aucs[np.isfinite(aucs)]
    macro = float(finite.mean()) if finite.size else float("nan")
    return macro, scores


def evaluate(model, loader, return_raw=False):
    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad(), torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
        for images, targets in loader:
            logits = model(images.to(DEVICE, non_blocking=True))
            all_logits.append(logits.float().cpu())
            all_targets.append(targets)
    logits, targets = torch.cat(all_logits), torch.cat(all_targets)
    macro, scores = calculate_auroc(logits, targets)
    return (macro, scores, (logits, targets)) if return_raw else (macro, scores)


def bootstrap_ci(raw, patient_ids, n_boot=200, seed=SEED):
    """95% CI for macro and per-label AUROC, resampling PATIENTS with replacement."""
    logits, targets = raw
    probs, targets = torch.sigmoid(logits).numpy(), targets.numpy()
    codes, uniques = pd.factorize(pd.Series(np.asarray(patient_ids)))
    order = np.argsort(codes, kind="stable")
    bounds = np.searchsorted(codes[order], np.arange(len(uniques) + 1))
    groups = [order[bounds[i]:bounds[i + 1]] for i in range(len(uniques))]

    rng = np.random.default_rng(seed)
    draws = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(groups), len(groups))
        idx = np.concatenate([groups[i] for i in pick])
        draws.append(per_label_auroc(probs[idx], targets[idx]))
    draws = np.array(draws)

    def interval(column):
        column = column[np.isfinite(column)]
        return (float("nan"), float("nan")) if column.size < 10 else tuple(np.percentile(column, [2.5, 97.5]))

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        macro_draws = np.nanmean(draws, axis=1)
    result = {"macro": interval(macro_draws)}
    for j, label in enumerate(LABELS):
        result[label] = interval(draws[:, j])
    return result


print("Models, BCE loss, AUROC and bootstrap helpers are ready.")

Models, BCE loss, AUROC and bootstrap helpers are ready.


In [11]:
# [Cell 11] Training function
def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


def train_model(backbone, config, epochs, train_frame, tune_frame,
                trial=None, save_path=None, patience=2):
    set_seed(SEED)   # same seed for every trial and backbone
    train_loader = make_loader(train_frame, config["batch_size"], training=True)
    tune_loader = make_loader(tune_frame, EVAL_BATCH_SIZE, training=False)
    model = build_model(backbone, config["dropout"]).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=config["learning_rate"] * 0.05
    )
    scaler = make_scaler()

    best_auroc, best_scores, best_epoch = -float("inf"), None, 0
    epochs_without_improvement = 0
    history = []
    pruned = False

    try:
        for epoch in range(epochs):
            model.train()
            start = time.perf_counter()
            running_loss, batches = 0.0, 0
            for images, targets in train_loader:
                images = images.to(DEVICE, non_blocking=True)
                targets = targets.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    logits = model(images)
                loss = bce_loss(logits.float(), targets)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                running_loss += float(loss.detach().cpu())
                batches += 1
            scheduler.step()

            val_auroc, per_finding = evaluate(model, tune_loader)
            if not np.isfinite(val_auroc):
                raise RuntimeError(
                    "Tuning macro AUROC is NaN: no label has >= MIN_POS positives and negatives "
                    "in the tuning frame. Increase the validation sample size or lower MIN_POS."
                )
            elapsed = time.perf_counter() - start
            history.append({
                "epoch": epoch + 1, "train_images": len(train_frame), "val_images": len(tune_frame),
                "train_loss": running_loss / max(batches, 1), "val_macro_auroc": val_auroc,
                "lr": optimizer.param_groups[0]["lr"], "seconds": elapsed,
            })
            print(f"{backbone} | epoch {epoch + 1}/{epochs} | train n={len(train_frame):,} val n={len(tune_frame):,} | "
                  f"loss={history[-1]['train_loss']:.4f} | val_macro_AUROC={val_auroc:.4f} | {elapsed:.0f}s")

            if val_auroc > best_auroc:
                best_auroc, best_scores, best_epoch = val_auroc, per_finding, epoch + 1
                epochs_without_improvement = 0
                if save_path is not None:
                    torch.save({"model_state_dict": model.state_dict(), "config": config,
                                "epoch": best_epoch, "backbone": backbone,
                                "tuning_macro_auroc": best_auroc}, save_path)
            else:
                epochs_without_improvement += 1

            if trial is not None:
                trial.report(val_auroc, epoch)
                if trial.should_prune():
                    pruned = True
                    break
            if epochs_without_improvement >= patience:
                print("Early stopping: tuning AUROC did not improve.")
                break
    finally:
        del model, optimizer, scheduler, scaler, train_loader, tune_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if pruned:
        raise optuna.TrialPruned()
    return {"macro_auroc": best_auroc, "per_finding_auroc": best_scores,
            "best_epoch": best_epoch, "config": config, "history": history}


print("train_model is ready.")

train_model is ready.


In [12]:
# [Cell 12] Optuna search per backbone
def make_objective(backbone):
    def objective(trial):
        config = {
            "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
            "batch_size": trial.suggest_categorical("batch_size", [16, 32]),
            "dropout": trial.suggest_categorical("dropout", [0.0, 0.2, 0.4]),
            "weight_decay": trial.suggest_float("weight_decay", 1e-3, 1e-1, log=True),
        }
        result = train_model(
            backbone, config, epochs=TUNING_EPOCHS, trial=trial,
            train_frame=search_train, tune_frame=search_val,
        )
        trial.set_user_attr("best_epoch", result["best_epoch"])
        trial.set_user_attr("per_finding_auroc", result["per_finding_auroc"])
        return result["macro_auroc"]
    return objective


studies, best_configs = {}, {}
for backbone in BACKBONES:
    print(f"\n{'=' * 65}\nTuning {backbone}: {N_TRIALS} trials, up to {TUNING_EPOCHS} epochs each\n{'=' * 65}")
    backbone_dir = OUTPUT_DIR / backbone
    backbone_dir.mkdir(parents=True, exist_ok=True)
    study = optuna.create_study(
        study_name=f"nih_{backbone}_{STUDY_TAG}",
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=4),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1),
        storage=f"sqlite:///{backbone_dir}/study_{STUDY_TAG}.db",
        load_if_exists=True,
    )
    remaining = max(0, N_TRIALS - len(study.trials))
    print(f"{len(study.trials)} trial(s) already stored; running {remaining} more.")
    study.optimize(make_objective(backbone), n_trials=remaining)
    studies[backbone] = study

    study.trials_dataframe().to_csv(backbone_dir / "optuna_trials.csv", index=False)
    best_configs[backbone] = dict(study.best_trial.params)
    (backbone_dir / "best_config.json").write_text(json.dumps(best_configs[backbone], indent=2))
    print(f"{backbone}: best search macro AUROC = {study.best_value:.4f}")
    print(json.dumps(best_configs[backbone], indent=2))

print("\nSearch complete for all backbones.")


Tuning densenet121: 8 trials, up to 4 epochs each


[I 2026-11-16 18:54:15,485] A new study created in RDB with name: nih_densenet121_v1


0 trial(s) already stored; running 8 more.
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 150MB/s]


densenet121 | epoch 1/4 | train n=10,000 val n=2,726 | loss=0.4342 | val_macro_AUROC=0.7690 | 176s
densenet121 | epoch 2/4 | train n=10,000 val n=2,726 | loss=0.3740 | val_macro_AUROC=0.7916 | 62s
densenet121 | epoch 3/4 | train n=10,000 val n=2,726 | loss=0.3472 | val_macro_AUROC=0.7970 | 62s
densenet121 | epoch 4/4 | train n=10,000 val n=2,726 | loss=0.3242 | val_macro_AUROC=0.7978 | 63s


[I 2026-11-16 19:00:19,355] Trial 0 finished with value: 0.7978402866965515 and parameters: {'learning_rate': 4.3284502212938785e-05, 'batch_size': 16, 'dropout': 0.0, 'weight_decay': 0.0013066739238053278}. Best is trial 0 with value: 0.7978402866965515.


... (49 outputs left out)

[I 2026-11-16 19:49:49,752] Trial 6 pruned. 


efficientnet_b0 | epoch 1/4 | train n=10,000 val n=2,726 | loss=0.5561 | val_macro_AUROC=0.5915 | 31s
efficientnet_b0 | epoch 2/4 | train n=10,000 val n=2,726 | loss=0.4460 | val_macro_AUROC=0.6387 | 30s


[I 2026-11-16 19:50:51,480] Trial 7 pruned. 


efficientnet_b0: best search macro AUROC = 0.7983
{
  "learning_rate": 0.0002962151658830348,
  "batch_size": 32,
  "dropout": 0.2,
  "weight_decay": 0.0026587543983272706
}

Search complete for all backbones.


In [13]:
# [Cell 13] Final retrain + test evaluation
if "best_configs" not in globals() or not best_configs:
    best_configs = {b: json.loads((OUTPUT_DIR / b / "best_config.json").read_text()) for b in BACKBONES}

summary_rows, per_label_rows, history_rows = [], [], []

for backbone in BACKBONES:
    print(f"\n{'=' * 65}\nFinal evaluation: {backbone}\n{'=' * 65}")
    backbone_dir = OUTPUT_DIR / backbone
    checkpoint_path = backbone_dir / "best_model.pt"
    config = best_configs[backbone]

    # 1) retrain best config on more data; best epoch is chosen on final_val
    retrain = train_model(
        backbone, config, epochs=FINAL_EPOCHS,
        train_frame=final_train, tune_frame=final_val, save_path=checkpoint_path,
    )
    for record in retrain["history"]:
        history_rows.append({"backbone": backbone, **record})

    # 2) reload the best checkpoint
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model = build_model(backbone, checkpoint["config"]["dropout"])
    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(DEVICE)

    # 3) official NIH test split, scored once, with patient-level bootstrap CIs
    test_loader = make_loader(test_data, EVAL_BATCH_SIZE, training=False)
    test_macro, test_scores, test_raw = evaluate(model, test_loader, return_raw=True)
    test_ci = bootstrap_ci(test_raw, test_data["patient_id"].to_numpy())

    summary_rows.append({
        "backbone": backbone,
        "search_best_macro_auroc": studies[backbone].best_value if "studies" in globals() and backbone in studies else float("nan"),
        "final_val_macro_auroc": retrain["macro_auroc"],
        "test_macro_auroc": test_macro,
        "test_ci_low": test_ci["macro"][0], "test_ci_high": test_ci["macro"][1],
        "best_epoch": checkpoint["epoch"],
        **config,
    })
    for label in LABELS:
        per_label_rows.append({
            "backbone": backbone, "label": label,
            "test_auroc": test_scores[label],
            "test_ci_low": test_ci[label][0], "test_ci_high": test_ci[label][1],
        })
    (backbone_dir / "test_results.json").write_text(json.dumps({
        "summary": summary_rows[-1], "test_per_label": test_scores,
        "test_ci": {k: list(v) for k, v in test_ci.items()},
    }, indent=2, default=float))

    print(f"{backbone}: test macro AUROC = {test_macro:.4f} "
          f"(95% CI {test_ci['macro'][0]:.4f} to {test_ci['macro'][1]:.4f})")
    del model, test_loader, checkpoint, test_raw
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summary_df = pd.DataFrame(summary_rows)
per_label_df = pd.DataFrame(per_label_rows)
history_df = pd.DataFrame(history_rows)
best_row = summary_df.loc[summary_df["final_val_macro_auroc"].idxmax()]
summary_df["selected_by_final_val"] = summary_df["backbone"].eq(best_row["backbone"])

summary_df.to_csv(OUTPUT_DIR / "architecture_comparison.csv", index=False)
per_label_df.to_csv(OUTPUT_DIR / "test_per_finding_auroc.csv", index=False)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)

print(f"\nSelected backbone (by final_val macro AUROC, not by test): {best_row['backbone']}")
display(summary_df.sort_values("final_val_macro_auroc", ascending=False))
print("\nPer-label test AUROC with 95% CI:")
display(per_label_df)
print("\nEpoch-wise history of the final retrains (sizes, loss, AUROC, time):")
display(history_df)


Final evaluation: densenet121
densenet121 | epoch 1/5 | train n=22,319 val n=2,726 | loss=0.3942 | val_macro_AUROC=0.7883 | 120s
densenet121 | epoch 2/5 | train n=22,319 val n=2,726 | loss=0.3624 | val_macro_AUROC=0.8026 | 118s
densenet121 | epoch 3/5 | train n=22,319 val n=2,726 | loss=0.3441 | val_macro_AUROC=0.8167 | 117s
densenet121 | epoch 4/5 | train n=22,319 val n=2,726 | loss=0.3184 | val_macro_AUROC=0.8213 | 116s
densenet121 | epoch 5/5 | train n=22,319 val n=2,726 | loss=0.2854 | val_macro_AUROC=0.8197 | 116s
densenet121: test macro AUROC = 0.7919 (95% CI 0.7850 to 0.7985)

Final evaluation: resnet50
resnet50 | epoch 1/5 | train n=22,319 val n=2,726 | loss=0.4023 | val_macro_AUROC=0.7740 | 98s
resnet50 | epoch 2/5 | train n=22,319 val n=2,726 | loss=0.3738 | val_macro_AUROC=0.7881 | 98s
resnet50 | epoch 3/5 | train n=22,319 val n=2,726 | loss=0.3572 | val_macro_AUROC=0.7993 | 99s
resnet50 | epoch 4/5 | train n=22,319 val n=2,726 | loss=0.3370 | val_macro_AUROC=0.8130 | 98s
r

,backbone,search_best_macro_auroc,final_val_macro_auroc,test_macro_auroc,test_ci_low,test_ci_high,best_epoch,learning_rate,batch_size,dropout,weight_decay,selected_by_final_val
0,densenet121,0.806727,0.821310,0.791929,0.784951,0.798505,4,0.000296,32,0.2,0.002659,True
1,resnet50,0.796925,0.818766,0.788906,0.782116,0.796426,5,0.000296,32,0.2,0.002659,False
2,efficientnet_b0,0.798313,0.815570,0.792206,0.785892,0.798927,3,0.000296,32,0.2,0.002659,False



Per-label test AUROC with 95% CI:


,backbone,label,test_auroc,test_ci_low,test_ci_high
0,densenet121,Effusion,0.790222,0.778791,0.803550
1,densenet121,Cardiomegaly,0.902437,0.885219,0.917975
2,densenet121,Pneumothorax,0.844068,0.820677,0.861233
3,densenet121,Atelectasis,0.742397,0.727236,0.759310
4,densenet121,Consolidation,0.706035,0.688517,0.722001
5,densenet121,Mass,0.791119,0.768538,0.805995
6,densenet121,Nodule,0.767223,0.748594,0.784431
7,resnet50,Effusion,0.795240,0.784886,0.808331
8,resnet50,Cardiomegaly,0.906230,0.888853,0.923207
9,resnet50,Pneumothorax,0.839436,0.817194,0.859289



Epoch-wise history of the final retrains (sizes, loss, AUROC, time):


,backbone,epoch,train_images,val_images,train_loss,val_macro_auroc,lr,seconds
0,densenet121,1,22319,2726,0.394243,0.788300,0.000269,119.668938
1,densenet121,2,22319,2726,0.362384,0.802643,0.000199,117.618286
2,densenet121,3,22319,2726,0.344076,0.816695,0.000112,116.615534
3,densenet121,4,22319,2726,0.318386,0.821310,0.000042,116.144082
4,densenet121,5,22319,2726,0.285369,0.819731,0.000015,116.246334
5,resnet50,1,22319,2726,0.402269,0.774010,0.000269,97.860278
6,resnet50,2,22319,2726,0.373771,0.788130,0.000199,97.998655
7,resnet50,3,22319,2726,0.357241,0.799300,0.000112,98.536862
8,resnet50,4,22319,2726,0.336955,0.812965,0.000042,98.395152
9,resnet50,5,22319,2726,0.311608,0.818766,0.000015,98.368182


In [14]:
# [Cell 14] Search trial summaries
for backbone in BACKBONES:
    t = pd.read_csv(OUTPUT_DIR / backbone / "optuna_trials.csv")
    cols = [c for c in ["number", "state", "value", "params_learning_rate", "params_batch_size",
                        "params_dropout", "params_weight_decay", "user_attrs_best_epoch"] if c in t.columns]
    t = t[cols].sort_values("value", ascending=False)
    done = t.loc[t["state"] == "COMPLETE", "value"]
    print(f"\n{backbone}: {len(t)} trials, {int((t['state'] == 'PRUNED').sum())} pruned | "
          f"best {done.max():.4f} | median {done.median():.4f} | gap {done.max() - done.median():.4f}")
    display(t)


densenet121: 8 trials, 5 pruned | best 0.8067 | median 0.7978 | gap 0.0089


,number,state,value,params_learning_rate,params_batch_size,params_dropout,params_weight_decay,user_attrs_best_epoch
1,1,COMPLETE,0.806727,0.000296,32,0.2,0.002659,4.0
0,0,COMPLETE,0.797840,0.000043,16,0.0,0.001307,4.0
6,6,PRUNED,0.792562,0.000085,32,0.4,0.006380,NaN
4,4,PRUNED,0.782616,0.000228,32,0.2,0.051632,NaN
5,5,PRUNED,0.781485,0.000313,32,0.2,0.001582,NaN
2,2,COMPLETE,0.768526,0.000020,32,0.0,0.016738,4.0
3,3,PRUNED,0.727646,0.000017,32,0.2,0.010677,NaN
7,7,PRUNED,0.720468,0.000016,32,0.2,0.001266,NaN



resnet50: 8 trials, 4 pruned | best 0.7969 | median 0.7931 | gap 0.0038


,number,state,value,params_learning_rate,params_batch_size,params_dropout,params_weight_decay,user_attrs_best_epoch
1,1,COMPLETE,0.796925,0.000296,32,0.2,0.002659,4.0
6,6,COMPLETE,0.794989,0.000085,32,0.4,0.006380,3.0
0,0,COMPLETE,0.791167,0.000043,16,0.0,0.001307,2.0
2,2,COMPLETE,0.788425,0.000020,32,0.0,0.016738,4.0
4,4,PRUNED,0.777576,0.000228,32,0.2,0.051632,NaN
5,5,PRUNED,0.776587,0.000313,32,0.2,0.001582,NaN
3,3,PRUNED,0.775913,0.000017,32,0.2,0.010677,NaN
7,7,PRUNED,0.771450,0.000016,32,0.2,0.001266,NaN



efficientnet_b0: 8 trials, 3 pruned | best 0.7983 | median 0.7947 | gap 0.0036


,number,state,value,params_learning_rate,params_batch_size,params_dropout,params_weight_decay,user_attrs_best_epoch
1,1,COMPLETE,0.798313,0.000296,32,0.2,0.002659,4.0
5,5,COMPLETE,0.796818,0.000313,32,0.2,0.001582,3.0
4,4,COMPLETE,0.794679,0.000228,32,0.2,0.051632,4.0
0,0,COMPLETE,0.778877,0.000043,16,0.0,0.001307,4.0
6,6,PRUNED,0.768093,0.000085,32,0.4,0.006380,NaN
2,2,COMPLETE,0.699505,0.000020,32,0.0,0.016738,4.0
3,3,PRUNED,0.647307,0.000017,32,0.2,0.010677,NaN
7,7,PRUNED,0.638703,0.000016,32,0.2,0.001266,NaN


In [15]:
# [Cell 15] Save split membership + zip everything for reuse in later sessions
import shutil

pd.concat(
    [d[["patient_id", "shard_path", "shard_index"]].assign(set=name)
     for name, d in split_sets.items()]
).to_csv(OUTPUT_DIR / "split_assignment.csv.gz", index=False)

zip_path = shutil.make_archive(str(OUTPUT_DIR.parent / "nih_tuning_checkpoint"), "zip",
                               root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name)
n_files = sum(1 for p in OUTPUT_DIR.rglob("*") if p.is_file())
print(f"{zip_path}: {Path(zip_path).stat().st_size / 1e6:.0f} MB, {n_files} files")

/kaggle/working/nih_tuning_checkpoint.zip: 129 MB, 19 files
